# QICK Pulse Demo
This notebook mirrors `python/demo_qick.py` but lays out each step explicitly. It assumes the QICK toolchain is available; cells will fail on machines without QICK hardware/software, so they are left unexecuted here.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from tempfile import NamedTemporaryFile

import qasmtrans as qt  # type: ignore
import qasmtrans.plot_pulses as plot_pulses  # type: ignore

REPO = Path(__file__).resolve().parents[1]
print(f"Project root: {REPO}")


## 1. Describe the two-qubit test program
This circuit matches the gates covered by the minimal backend/pulse template and will be used throughout the demo.


In [ ]:
qasm_text = """OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
x q[0];
cx q[0],q[1];
rz(1.5708) q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""
print(qasm_text)


## 2. Build lightweight backend and pulse template JSON blobs
Temporary files keep the workflow self-contained. Adjust the dictionaries below to represent your hardware.


In [ ]:
backend_doc = {
    "name": "demo_backend",
    "version": "0.1",
    "num_qubits": 2,
    "basis_gates": ["x", "rz", "cx"],
    "gate_lens": {"x0": 1.0, "x1": 1.0, "rz0": 0.0, "rz1": 0.0, "cx0_1": 2.0},
    "gate_errs": {"x0": 0.0, "x1": 0.0, "rz0": 0.0, "rz1": 0.0, "cx0_1": 0.0},
    "cx_coupling": ["0_1"],
}

template_doc = {
    "name": "demo_template",
    "version": "0.1",
    "pulse_definitions": [
        {"id": "rz_q0", "gate": "rz", "qubits": [0], "shape": "virtual", "waveform_type": "virtual", "width": 0.0, "amplitude": 0.0, "virtual": True},
        {"id": "rz_q1", "gate": "rz", "qubits": [1], "shape": "virtual", "waveform_type": "virtual", "width": 0.0, "amplitude": 0.0, "virtual": True},
        {"id": "x_q0", "gate": "x", "qubits": [0], "shape": "const", "waveform_type": "const", "width": 1e-6, "amplitude": 1.0},
        {"id": "x_q1", "gate": "x", "qubits": [1], "shape": "const", "waveform_type": "const", "width": 1e-6, "amplitude": 1.0},
        {"id": "cx_q0_q1", "gate": "cx", "qubits": [0, 1], "shape": "const", "waveform_type": "const", "width": 2e-6, "amplitude": 1.0},
    ],
}

backend_tmp = NamedTemporaryFile("w", suffix=".json", delete=False)
template_tmp = NamedTemporaryFile("w", suffix=".json", delete=False)
backend_path = Path(backend_tmp.name)
template_path = Path(template_tmp.name)
backend_tmp.close()
template_tmp.close()
backend_path.write_text(json.dumps(backend_doc, indent=2))
template_path.write_text(json.dumps(template_doc, indent=2))
print("Backend config ->", backend_path)
print("Pulse template ->", template_path)


## 3. Transpile to pulses via QASMTrans
Outputs land under `data/output_demo_qick`. Set `allow_parameterized_merge`/`limited_qubits` to match your use case.


In [ ]:
opts = qt.TranspileOptions()
opts.backend_config = str(backend_path)
opts.pulse_template_path = str(template_path)
opts.output_path = str(REPO / "data" / "output_demo_qick")
opts.allow_parameterized_merge = True
opts.limited_qubits = True

result = qt.transpile_qasm(qasm_text, opts)
print("Log:", result.log)
print("Pulse JSON path:", result.pulse_path)


## 4. Plot the generated pulse schedule
The helper below writes a PNG file so you can inspect the analog and virtual events without a GUI.


In [ ]:
pulse_doc = result.pulse_doc if result.pulse_doc is not None else json.loads(result.pulse_schedule or "{}")
analog_events, virtual_events, labels = plot_pulses.build_qubit_events(pulse_doc)
plot_path = REPO / "data" / "output_demo_qick_pulses.png"
plot_pulses.plot_events(
    analog_events_by_qubit=analog_events,
    virtual_events_by_qubit=virtual_events,
    label_sequences=labels,
    title="QASMTrans Pulse Schedule",
    output_path=plot_path,
    dpi=150,
    samples_per_us=200.0,
)
print(f"Pulse plot saved to {plot_path}")


## 5. Emit a QICK program (optional)
`qt.emit_qick` requires the QICK stack. Leave `run_enabled=False` to produce a summary without touching hardware.


In [ ]:
qick_cfg = {"qubit_gen_map": {"0": 0, "1": 1}, "pulse_freq": 6.0e9}
try:
    summary = qt.emit_qick(pulse_doc, qick_cfg, run_enabled=False, summary_only=True)
    print("emit_qick summary (no hardware run):", json.dumps(summary, indent=2))
except ImportError as exc:
    print("QICK not available on this machine:", exc)
except Exception as exc:
    print("emit_qick failed:", exc)
finally:
    backend_path.unlink(missing_ok=True)
    template_path.unlink(missing_ok=True)


That completes the QICK-oriented walkthrough. Update any of the cells above to experiment with different circuits, hardware parameters, or emission settings.
